In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm
import pandas as pd
from math import prod

from cnn_surgery.lenses.regressor_lens import get_regressor_lens, mse_mae
from cnn_surgery.utils.evaluate_per_class_accuracy import evaluate_classifier, load_testset_data
from cnn_surgery.utils.load_dataset import load_dataset, load_multi_stage_dataset
from cnn_surgery.utils.reconstruct_network import reconstruct_network, SHAPES
from cnn_surgery.utils.metrics import clipped_negative_mean_difference

from cnn_surgery.utils.reconstruct_network import SHAPES
DATASET = "fashion_mnist"
LAYERS_TO_KEEP = [
    "sequential/dense/bias:0",
    "sequential/dense/kernel:0",
]

def test_network_accuracy(weights: np.ndarray | torch.Tensor, activation_fn):
    """
    Reconstructs a network from weights and activation function, evaluates it on the test set
    BEWARE: when using it in unlearning, the input could be a tensor, so be sure to convert it to numpy array first:
        weights = weights.detach().numpy()

    Returns:
        mean accuracy: float
        per class accuracies: list of floats
    """

    if isinstance(weights, torch.Tensor):
        weights = weights.detach().numpy()

    CNNModel = reconstruct_network(weights, activation_fn)
    # this returns a Keras model (Unterthiner code), we have to compile it first
    CNNModel.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    x_test, y_test = load_testset_data(DATASET)
    overall_acc, class_accs = evaluate_classifier(CNNModel, x_test, y_test)
    return overall_acc, class_accs

TODO: vergelijk test_network_accuracy() met evaluate_classifier(). Als ze hetzelfde doen, kan test_network_accuracy() weg.

Load dataset and train metanetwork on a subset of the weights. 
how are the weights distributed across the layers? check reconstruct network functions.

In [ ]:
train, val, _ = load_multi_stage_dataset(dataset=DATASET).values()

full_weights_train = train[0]
full_weights_val = val[0]

accuracies_train = train[1]
accuracies_val = val[1]

configs_train = train[2]
configs_val = val[2]

def split_weights(weights: np.ndarray, layers_to_keep: list[str]) -> np.ndarray:
    "Returns a new array of the weights where only the specified layers are kept, others are removed"

    assert all(layer in SHAPES.keys() for layer in layers_to_keep), f"One or more of the specified layers {layers_to_keep} is not in SHAPES keys {list(SHAPES.keys())}"
    # assert if the order of the layers_to_keep is the same as in SHAPES
    shapes_in_order = [layer for layer in SHAPES.keys() if layer in layers_to_keep]
    assert layers_to_keep == shapes_in_order[:len(layers_to_keep)], f"The order of the specified layers {layers_to_keep} is not the same as how they appear in SHAPES: {shapes_in_order}"

    idx = 0
    weights_of_layers_to_keep = []
    for layer_name, layer_shape in SHAPES.items():
        layer_size = prod(layer_shape)
        layer_weights = weights[idx : idx + layer_size]
        idx += layer_size
        if layer_name in layers_to_keep:
            # keep the layer weights
            weights_of_layers_to_keep.append(layer_weights)

    new_weights = np.concatenate(weights_of_layers_to_keep)

    assert new_weights.shape[0] == sum(prod(SHAPES[layer]) for layer in layers_to_keep), f"New weights shape {new_weights.shape[0]} does not match expected shape {sum(prod(SHAPES[layer]) for layer in layers_to_keep)}"

    return new_weights

def unsplit_weights(full_weights, isolated_weights, layers_that_were_isolated):
    "Returns a full array of the weights where the isolated weights are put back into their original positions"

    idx_full = 0
    idx_isolated = 0
    reconstructed_weights = []
    for layer_name, layer_shape in SHAPES.items():
        layer_size = prod(layer_shape)
        if layer_name in layers_that_were_isolated:
            # take weights from isolated_weights
            layer_weights = isolated_weights[idx_isolated : idx_isolated + layer_size]
            idx_isolated += layer_size
        else:
            # take weights from full_weights
            layer_weights = full_weights[idx_full : idx_full + layer_size]
        reconstructed_weights.append(layer_weights)
        idx_full += layer_size
    new_weights = np.concatenate(reconstructed_weights)
    assert new_weights.shape[0] == full_weights.shape[0], f"Reconstructed weights shape {new_weights.shape[0]} does not match full weights shape {full_weights.shape[0]}"
    return new_weights

# apply split_weights to all weights arrays (axis=0) in weights_train and weights_val
isolated_weights_train = np.array([split_weights(weights, LAYERS_TO_KEEP) for weights in full_weights_train])
isolated_weights_val = np.array([split_weights(weights, LAYERS_TO_KEEP) for weights in full_weights_val])

Train a metanetwork with these subsets of the weights.

In [ ]:
MetaNetwork = get_regressor_lens(isolated_weights_train, accuracies_train, isolated_weights_val, accuracies_val, device="cpu")

## Testing some stuff

In [ ]:
print(full_weights_train.shape)
print(full_weights_val.shape)
print(isolated_weights_train.shape)
print(isolated_weights_val.shape)

In [ ]:
MODEL_IDX = 18354

print(isolated_weights_val[MODEL_IDX].shape)
input_tensor = torch.tensor(isolated_weights_val[MODEL_IDX], requires_grad=True, dtype=torch.float32)
preds = MetaNetwork(input_tensor.unsqueeze(0)).squeeze(0).detach().numpy() # type: ignore
actual_accs = accuracies_val[MODEL_IDX]

classes = np.arange(len(preds))
bar_width = 0.25
plt.bar(classes,           actual_accs, width=bar_width, label="Actual", alpha=0.7)
plt.bar(classes + bar_width,  preds,       width=bar_width, label="Predicted", alpha=0.7)
plt.xlabel("Class index")
plt.ylabel("Accuracy")
plt.title("Class-wise accuracies: Predicted vs. Actual")
plt.xticks(classes)
plt.ylim(0, 1)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.legend()

Seems to work. Now: 
## Unlearning

In [ ]:
from IPython.display import clear_output, display
from matplotlib import pyplot as plt
MODEL_IDX = 18354

def boost_loss(pred, target_idx, beta=1.0):
    target_term = pred[target_idx]  # we want to minimize this

    # set target index zero for this term by multiplying with a mask
    mask = torch.ones_like(pred, requires_grad=False)
    mask[target_idx] = 0
    maintain_rest_term = (pred * mask).sum()  # we want to increase or at least maintain the accuracy of the other classes

    return target_term - beta * maintain_rest_term

assert isinstance(MetaNetwork, nn.Module), f"MetaNetwork is not an instance of torch.nn.Module but {type(MetaNetwork)}" # to stop linter nagging
full_input_weights: np.ndarray = full_weights_val[MODEL_IDX]
isolated_input_weights: np.ndarray = isolated_weights_val[MODEL_IDX]
steps = 100
step_size = 1
target_class = 5
og_config: pd.Series = configs_val.iloc[MODEL_IDX]
beta = 0.1
stop_threshold = 0.8
animate = True

print("Starting unlearning procedure")
diffs_list = []
loss_list = []
target_term_list = []
# mse_term_list = []
mae_term_list = []
cosine_sim_list = []
metric_list = []
maintain_rest_list = []

before_accs = test_network_accuracy(full_input_weights, og_config["config.activation"])[1]

# convert input weights to tensor for optimization
doctored_input_weights = torch.tensor(isolated_input_weights, requires_grad=True, dtype=torch.float32)
MetaNetwork.eval()

# optimize only the INPUT weights
for param in MetaNetwork.parameters():
    param.requires_grad = False

# for i in tqdm(range(steps), desc="Unlearning in progress"):
for i in range(steps): 

    # this is expensive: a new model needs to be reconstructed and evaluated at every step
    full_doctored_weights = unsplit_weights(full_input_weights, doctored_input_weights.detach().numpy(), LAYERS_TO_KEEP)
    true = torch.tensor(
        test_network_accuracy(full_doctored_weights, og_config["config.activation"])[1],
        dtype=torch.float32,
    )

    pred: torch.Tensor = MetaNetwork(doctored_input_weights.unsqueeze(0)).squeeze(0)  # forward pass
    loss = boost_loss(pred, target_class, beta=beta)
    loss.backward()  # compute gradients

    gradient = doctored_input_weights.grad.clone() # type: ignore
    if i == 0:
        first_gradient = gradient

    ### ACTUAL OPTIMIZATION STEP ###
    with torch.no_grad():
        doctored_input_weights -= step_size * gradient  # gradient step # type: ignore
        doctored_input_weights.grad.zero_()  # zero gradients # type:  ignore

    # stopping criterion: cosine_similarity between gradient and first gradient should stay below a threshold
    cos_sim = np.dot(gradient, first_gradient) / (np.linalg.norm(gradient) * np.linalg.norm(first_gradient))
    if cos_sim < stop_threshold:
        print(f"Stopping early at step {i} due cosine similarity below threshold ({cos_sim:.4f} < {stop_threshold})")
        break

    # -------- for analysis --------
    target_term: float = pred[target_class].item()
    # mse_term: float = ((true - pred) ** 2).mean().item()
    mae_term: float = (true - pred).abs().mean().item()
    mean_diff = abs((pred.detach().numpy() - np.array(true.detach().numpy()))).mean()
    metric = clipped_negative_mean_difference(np.array(before_accs), true.detach().numpy(), target_class)
    mask = torch.ones_like(pred, requires_grad=False)
    mask[target_class] = 0
    maintain_rest_term = beta * -1 * (pred * mask).sum().item()

    loss_list.append(loss.item())
    target_term_list.append(target_term)
    # mse_term_list.append(mse_term)
    mae_term_list.append(mae_term)
    diffs_list.append(mean_diff)
    cosine_sim_list.append(cos_sim)
    metric_list.append(metric)
    maintain_rest_list.append(maintain_rest_term)

    if animate:
        # ------- plot -------
        clear_output(wait=True)
        fig, axes = plt.subplots(2, 2, figsize=(18, 12))

        before_accs = accuracies_val[MODEL_IDX]
        assert isinstance(MetaNetwork, nn.Module), f"MetaNetwork is not an instance of torch.nn.Module but {type(MetaNetwork)}"
        preds = MetaNetwork(doctored_input_weights.unsqueeze(0)).squeeze().detach().numpy()

        # actual_accs = test_network_accuracy(doctored_input_weights.detach().numpy(), configs_val.iloc[MODEL_IDX]["config.activation"])[1]
        actual_accs = true

        classes = np.arange(len(preds))
        bar_width = 0.25
        ax = axes[0, 0]
        ax.bar(classes - bar_width, before_accs, width=bar_width, label="Before unlearning", alpha=0.7)
        ax.bar(classes,              actual_accs, width=bar_width, label="After unlearning", alpha=0.7)
        ax.bar(classes + bar_width,  preds,       width=bar_width, label="Predicted by RegressorLens", alpha=0.7)
        ax.set_xlabel("Class index")
        ax.set_ylabel("Accuracy")
        ax.set_title(f"Step {i}: Class-wise accuracies: Before vs Predicted & Actual")
        ax.set_xticks(classes)
        ax.set_ylim(0, 1)
        ax.grid(axis="y", linestyle="--", alpha=0.7)
        ax.legend()

        # ------- (2,0) unlearning metric -------
        ax = axes[0, 1]
        ax.plot(metric_list)
        ax.set_xlabel("Unlearning step")
        ax.set_ylabel("Unlearning metric")
        ax.set_title("Unlearning metric over unlearning steps")
        ax.set_ylim(-1.05, 1.05)
        ax.grid(linestyle="--", alpha=0.3)

        ax = axes[1, 0]
        ax.plot(diffs_list)
        ax.set_xlabel("Unlearning step")
        ax.set_ylabel("Mean prediction error")
        ax.set_title("Mean prediction error over unlearning steps")
        ax.set_ylim(-0.05, 1.05)
        ax.grid(linestyle="--", alpha=0.3)

        ax = axes[1, 1]
        ax.plot(cosine_sim_list)
        ax.set_xlabel("Unlearning step")
        ax.set_ylabel("Cosine similarity")
        ax.set_title("Cosine similarity with first gradient")
        ax.set_ylim(0, 1.05)
        ax.grid(linestyle="--", alpha=0.3)

        fig.suptitle(f"Unlearning procedure for model index {MODEL_IDX}, target class {target_class}", fontsize=16, fontweight='bold')
        plt.tight_layout(rect=[0, 0, 1, 0.97]) # type: ignore

        display(fig)

        # input("Press Enter to continue to the next step...")

        plt.close(fig)